In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nabeelqureshitiii/student-performance-dataset/student_performance.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/nabeelqureshitiii/student-performance-dataset/student_performance.csv')
df.head()

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score,grade
0,1,18.5,95.6,3.8,97.9,A
1,2,14.0,80.0,2.5,83.9,B
2,3,19.5,86.3,5.3,100.0,A
3,4,25.7,70.2,7.0,100.0,A
4,5,13.4,81.9,6.9,92.0,A


In [3]:
df.shape

(1000000, 6)

In [4]:
df.describe()

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,total_score
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,500000.500000,15.029127,84.711046,5.985203,84.283845
std,288675.278933,6.899431,9.424143,1.956421,15.432969
min,1.000000,0.000000,50.000000,0.000000,9.400000
25%,250000.750000,10.300000,78.300000,4.700000,73.900000
50%,500000.500000,15.000000,85.000000,6.000000,87.500000
75%,750000.250000,19.700000,91.800000,7.300000,100.000000
max,1000000.000000,40.000000,100.000000,10.000000,100.000000


In [5]:
df.isnull().sum()

student_id                 0
weekly_self_study_hours    0
attendance_percentage      0
class_participation        0
total_score                0
grade                      0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 6 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   student_id               1000000 non-null  int64  
 1   weekly_self_study_hours  1000000 non-null  float64
 2   attendance_percentage    1000000 non-null  float64
 3   class_participation      1000000 non-null  float64
 4   total_score              1000000 non-null  float64
 5   grade                    1000000 non-null  object 
dtypes: float64(4), int64(1), object(1)
memory usage: 45.8+ MB


In [7]:
X = df.drop(columns='total_score')
y = df['total_score']

In [8]:
print("shape :",X.shape)

shape : (1000000, 5)


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder


In [10]:
encoder = OrdinalEncoder(categories=[['A','B','C','D','F']])
X['grade'] = encoder.fit_transform(X[['grade']]) + 1

In [11]:
X['grade'].value_counts()

grade
1.0    548644
2.0    258174
3.0    141980
4.0     44998
5.0      6204
Name: count, dtype: int64

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)
X_train.shape

(800000, 5)

In [13]:
X_train

,student_id,weekly_self_study_hours,attendance_percentage,class_participation,grade
566853,566854,23.0,78.3,5.1,1.0
382311,382312,16.9,98.2,4.8,1.0
241519,241520,10.8,93.1,5.3,3.0
719220,719221,13.2,94.7,3.8,1.0
905718,905719,14.0,88.4,7.7,2.0
...,...,...,...,...,...
259178,259179,8.2,83.4,5.5,2.0
365838,365839,10.5,86.7,6.7,3.0
131932,131933,11.5,73.5,7.2,1.0
671155,671156,12.9,90.1,6.5,1.0


In [14]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [15]:
y_pred = lr.predict(X_test)
y_pred

array([48.32541559, 81.4566634 , 77.05679216, ..., 98.50491951,
       33.2163015 , 93.09506585])

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [17]:
y_test.shape

(200000,)

In [18]:
print("MSE:", mean_squared_error(y_pred,y_test))
print("MAE:", mean_absolute_error(y_pred,y_test))
print("R2:", r2_score(y_pred,y_test))

MSE: 16.572453319567366
MAE: 3.3843689762798395
R2: 0.9251469575663064


## Lets make our own regression class

In [19]:
class MulitReg:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X_train,y_train):
        X_train = np.insert(X_train,0,1, axis=1)
        betas = np.linalg.inv(X_train.T @ X_train) @ X_train.T @ y_train
        self.coef_ = betas[1:]
        self.intercept_ = betas[0]
        
    def predict(self, X_test):
        y_pred = self.intercept_ + X_test @ self.coef_
        return y_pred
        
    def evaluation(self, y_test):
        print("MSE:", mean_squared_error(y_pred,y_test))
        print("MAE:", mean_absolute_error(y_pred,y_test))
        print("R2:", r2_score(y_pred,y_test))

In [20]:
lr = MulitReg()
lr.fit(X_train, y_train)

In [21]:
lr.predict(X_test)

987231    48.325416
79954     81.456663
567130    77.056792
500891    93.471790
55399     33.635355
            ...    
90245     94.872271
639296    94.046297
311939    98.504920
324459    33.216302
390499    93.095066
Length: 200000, dtype: float64

In [22]:
lr.evaluation(y_test)

MSE: 16.572453319567366
MAE: 3.3843689762798395
R2: 0.9251469575663064


### Observation : Now we see that our own class values and the sklearn model values are same
`MSE: 16.572453319567366`
`MAE: 3.3843689762798395`
`R2: 0.9251469575663064`